# 라이브러리 호출

## 라이브러리 설치

In [ ]:
!pip install flask flask-ngrok

In [ ]:
!pip install pyngrok

## 호출 및 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/여친_TravelMate

In [13]:
import pandas as pd
import numpy as np
from datetime import datetime

from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import pickle

from flask import Flask, jsonify, request,render_template
from flask_ngrok import run_with_ngrok
from pyngrok import ngrok
import requests
from Course_recommendation_v9 import TourConstructionHeuristics as Course_algs

# 필요한 데이터 및 모델 로드

수정이 잦아서 df를 생성하는게 편함

가능하다면 data에 평균 체류시간 넣어서 merge하면 따로 작업할 필요 없음

In [ ]:
df = pd.read_csv("Data/Final/final_5.csv")

## 좌표, 주소 데이터 전처리

In [ ]:
# place_check = df[['VISIT_AREA_NM', 'VISIT_AREA_MAIN_TYPE', 'VISIT_COUNT', 'VISIT_REASON',
#        'RES_CAFE']]
# address_check = df[['VISIT_AREA_NM', 'X_COORD', 'Y_COORD', 'ADDR']]

# place_check.drop_duplicates(inplace = True)
# place_check.reset_index(drop = True, inplace = True)

# address_check.drop_duplicates(inplace = True)
# address_check.reset_index(drop = True, inplace = True)

In [ ]:
# place_check.info()

In [ ]:
# address_check.info()

In [ ]:
# address_check['VISIT_AREA_NM'].value_counts()

In [ ]:
# address_check[address_check['VISIT_AREA_NM'] == '천지연폭포']

In [ ]:
# address_check.drop([187, 188, 189, 190], axis = 0, inplace = True)
# address_check[address_check['VISIT_AREA_NM'] == '천지연폭포']

In [ ]:
# # 각 'VISIT_AREA_NM' 그룹에서 'ADDR'의 최빈값을 구함
# mode_df = address_check.groupby('VISIT_AREA_NM')['ADDR'].agg(lambda x: x.mode()[0]).reset_index()

# # 기존의 df와 최빈값을 구한 mode_df를 VISIT_AREA_NM과 ADDR를 기준으로 병합
# result = pd.merge(address_check, mode_df, on=['VISIT_AREA_NM', 'ADDR'])

# # 중복된 행을 제거하고 필요한 컬럼만 남기기
# result = result.drop_duplicates(subset=['VISIT_AREA_NM'])

In [ ]:
# result.reset_index(drop = True, inplace = True)
# result

In [ ]:
# result.to_csv('Data/Final/final_05_address.csv', index = False)

In [ ]:
# address_check.groupby('VISIT_AREA_NM')

## 방문지 데이터 호출

In [ ]:
places = pd.read_csv("Data/Final/final_05_place.csv")
address = pd.read_csv("Data/Final/final_05_address.csv")

### 데이터 확인

In [ ]:
places.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7598 entries, 0 to 7597
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   VISIT_AREA_NM         7598 non-null   object
 1   VISIT_AREA_MAIN_TYPE  7598 non-null   int64 
 2   VISIT_COUNT           7598 non-null   int64 
 3   VISIT_REASON          7598 non-null   object
 4   RES_CAFE              7598 non-null   int64 
dtypes: int64(3), object(2)
memory usage: 296.9+ KB


In [ ]:
places.columns

Index(['VISIT_AREA_NM', 'VISIT_AREA_MAIN_TYPE', 'VISIT_COUNT', 'VISIT_REASON',
       'RES_CAFE'],
      dtype='object')

In [ ]:
address.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7598 entries, 0 to 7597
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   VISIT_AREA_NM  7598 non-null   object 
 1   X_COORD        7598 non-null   float64
 2   Y_COORD        7598 non-null   float64
 3   ADDR           7598 non-null   object 
dtypes: float64(2), object(2)
memory usage: 237.6+ KB


In [ ]:
address.columns

Index(['VISIT_AREA_NM', 'X_COORD', 'Y_COORD', 'ADDR'], dtype='object')

## 모델 호출

In [ ]:
loaded_model = load_model('./Model/Fitted_NCF_0/fitted_ncf_best_model_00.keras')

# 장소 추천 함수

## Flutter Input Features

In [ ]:
t_features = [
    'TRAVEL_MISSION_PRIORITY', # 개별 미션 최우선 순위 - c
    'TRAVEL_STYL_1', # 자연, 도시
    'TRAVEL_STYL_5', # 휴양/휴식, 체험활동
    'TRAVEL_STYL_6', # 유명지
    'TRAVEL_MOTIVE_1', # 여행 동기
    ]

## 사용자 정의 함수

### /list_recommend 함수

In [ ]:
# 전처리 객체 불러오기 함수
def load_preprocessors():
    with open('Model/Fitted_NCF_0/user_scaler_1.pkl', 'rb') as f:
        user_scaler = pickle.load(f)
    with open('Model/Fitted_NCF_0/item_scaler_1.pkl', 'rb') as f:
        item_scaler = pickle.load(f)
    with open('Model/Fitted_NCF_0/user_encoder_1.pkl', 'rb') as f:
        user_encoder = pickle.load(f)
    with open('Model/Fitted_NCF_0/item_encoder_1.pkl', 'rb') as f:
        item_encoder = pickle.load(f)
    return user_scaler, item_scaler, user_encoder, item_encoder

In [ ]:
# 연속형 데이터 스케일링 함수
def min_max_scaling(df, scaler, columns):
    df[columns] = scaler.transform(df[columns])
    return df

# 범주형 데이터 인코딩 함수
def one_hot_encode_columns(df, encoder, columns):
    # 인코딩할 컬럼이 데이터프레임에 모두 존재하는지 확인
    if not all(col in df.columns for col in columns):
        missing_cols = [col for col in columns if col not in df.columns]
        raise ValueError(f"다음 컬럼이 데이터프레임에 존재하지 않습니다: {missing_cols}")

    # 인코딩된 데이터프레임 생성
    encoded_array = encoder.transform(df[columns])
    encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(columns), index=df.index)

    # 원래 데이터프레임에서 인코딩된 컬럼 제거 후 인코딩된 컬럼 추가
    df = df.drop(columns=columns)
    df = pd.concat([df, encoded_df], axis=1)

    return df

In [ ]:
# INPUT 전처리 함수 정의
def preprocess_input_data(df):
    user_features = ['TRAVEL_STYL_1', 'TRAVEL_STYL_5', 'TRAVEL_STYL_6', 'TRAVEL_MOTIVE_1', 'TRAVEL_MISSION_PRIORITY']
    item_features = ['VISIT_COUNT', 'VISIT_AREA_MAIN_TYPE', 'VISIT_REASON']

    # user_data와 item_data 분리
    user_data = df[user_features].copy()
    item_data = df[item_features].copy()

    # 전처리 객체 불러오기
    user_scaler, item_scaler, user_encoder, item_encoder = load_preprocessors()

    # 연속형 데이터 처리
    for col in ['TRAVEL_STYL_1', 'TRAVEL_STYL_5', 'TRAVEL_STYL_6']:
        # 각 컬럼에 대해 수동으로 스케일링 적용
        min_val = user_scaler.data_min_[user_data.columns.get_loc(col)]
        max_val = user_scaler.data_max_[user_data.columns.get_loc(col)]

        # Min-Max Scaling 공식 적용
        user_data[col] = (user_data[col] - min_val) / (max_val - min_val)

    # item_data에 대한 스케일링
    min_val_item = item_scaler.data_min_[item_data.columns.get_loc('VISIT_COUNT')]
    max_val_item = item_scaler.data_max_[item_data.columns.get_loc('VISIT_COUNT')]
    item_data['VISIT_COUNT'] = (item_data['VISIT_COUNT'] - min_val_item) / (max_val_item - min_val_item)

    # 범주형 데이터 처리
    user_cat_columns = ['TRAVEL_MOTIVE_1', 'TRAVEL_MISSION_PRIORITY']
    user_data = one_hot_encode_columns(user_data, user_encoder, user_cat_columns)

    item_cat_columns = ['VISIT_AREA_MAIN_TYPE']
    item_data = one_hot_encode_columns(item_data, item_encoder, item_cat_columns)

    # VISIT_REASON이 문자열이 아닌 리스트 또는 벡터 형태로 존재하는지 확인
    if isinstance(item_data['VISIT_REASON'].iloc[0], str):
        # VISIT_REASON이 문자열로 저장되어 있다면, 이를 리스트로 변환 (필요한 경우)
        item_data['VISIT_REASON'] = item_data['VISIT_REASON'].apply(lambda x: eval(x) if isinstance(x, str) else x)

    # VISIT_REASON 벡터를 VISIT_REASON_1, VISIT_REASON_2, ...으로 분리하여 데이터프레임으로 변환
    visit_reason_columns = [f'VISIT_REASON_{i+1}' for i in range(len(item_data['VISIT_REASON'].iloc[0]))]
    visit_reason_df = pd.DataFrame(item_data['VISIT_REASON'].tolist(), columns=visit_reason_columns)

    # Join, Drop
    item_data = item_data.join(visit_reason_df)
    item_data = item_data.drop('VISIT_REASON', axis = 1)

    return user_data, item_data

In [ ]:
def predict_result(model, input):
    # 전처리
    user_input, item_input = preprocess_input_data(input)

    # 예측
    prediction = model.predict([user_input, item_input])
    return prediction

In [ ]:
def recommend(input, places):
  features = ['VISIT_AREA_MAIN_TYPE', 'VISIT_COUNT','VISIT_REASON', 'TRAVEL_MISSION_PRIORITY', 'TRAVEL_STYL_1', 'TRAVEL_STYL_5', 'TRAVEL_STYL_6', 'TRAVEL_MOTIVE_1', 'VISIT_AREA_NM']
  m_var = ['VISIT_AREA_MAIN_TYPE', 'VISIT_COUNT', 'VISIT_AREA_NM', 'SCORE']

  # 여행자 정보 dict 생성
  traveler = dict(zip(t_features, input))

  print("Traveler Load  ... (1) Complete! \n")

  df = places[['VISIT_AREA_NM', 'VISIT_COUNT', 'VISIT_REASON', 'VISIT_AREA_MAIN_TYPE']]
  for col, value in traveler.items():
    df[col] = value

  print("Concat Data ... (2) Complete! \n")

  # 모델에 맞춰 column 정렬
  try:
    df = df[features]
  except KeyError as e:
    print("KeyError occurred while reordering columns:", e)
    raise

  print("Sort Features ... (3) Complete! \n")

  # 예측값 추가(모델)
  df['SCORE'] = predict_result(loaded_model, df)

  print("Model Process ... (4) Complete! \n")

  # SCORE를 기준으로 정렬
  sort_list = df.sort_values('SCORE', ascending=False)
  sort_list.reset_index(drop=True, inplace=True)

  print("Sorting ... (5) Complete! \n")

  # places와 병합
  merge = pd.merge(sort_list, places, on=['VISIT_AREA_NM', 'VISIT_AREA_MAIN_TYPE', 'VISIT_COUNT'], how='left')

  print("Merge Model with Place Data ... (6) Complete! \n")

  # 1. 숙소(VIS = 24)
  lodging = merge[merge['VISIT_AREA_MAIN_TYPE'] == 24]
  lod_list = lodging[m_var]
  lod_40 = lod_list[:40]

  # 식당 카페(VIS = 11)
  rescafe = merge[merge['VISIT_AREA_MAIN_TYPE'] == 11]

  # 2. 식당(VIS = 11, RES_CAFE = 0)
  res = rescafe[rescafe['RES_CAFE'] == 0]
  res_list = res[m_var]
  res_40 = res_list[:40]

  # 3. 카페(VIS = 11, RES_CAFE = 1)
  caf = rescafe[rescafe['RES_CAFE'] == 1]
  caf_list = caf[m_var]
  caf_40 = caf_list[:40]

  # 4. 여행지(VIS != 24 && VIS != 11)
  trav = merge[(merge['VISIT_AREA_MAIN_TYPE'] != 24) & (merge['VISIT_AREA_MAIN_TYPE'] != 11)]
  trav_list = trav[m_var]
  trav_40 = trav_list[:40]

  result = pd.concat([lod_40, res_40, caf_40, trav_40], axis=0)
  result.reset_index(drop=True, inplace=True)

  print("Classify ... (7) Complete! \n")

  merge = pd.merge(result, address, on = 'VISIT_AREA_NM', how = 'left')

  print("Add Coordinate ... (8) Complete! \n")

  return merge

### /course_recommend 함수

In [ ]:
# 이름 변경 함수
def rename_df(df):
  df.rename(columns = {
    'VISIT_AREA_NM': 'name',
    'ADDR': 'address',
    'SCORE': 'satisfaction',
    'Y_COORD': 'latitude',
    'X_COORD': 'longitude',
    'START': 'start',
    'END': 'end'
  }, inplace = True)

  return df

# 최종 출력 함수
def sort_by_day(day, df):
  day_sort = df[df['DAY'] == day]

  # 'DAY' 열 삭제
  day_sort = day_sort.drop('DAY', axis = 1)

  # 'ORDER' 열을 기준으로 정렬
  day_sorted = day_sort.sort_values(by='ORDER')

  # 불필요한 열 삭제
  day_sorted.drop(['ORDER', 'VISIT_AREA_MAIN_TYPE', 'VISIT_COUNT', 'AVG_TIME_MIN'], axis = 1, inplace = True)

  renamed = rename_df(day_sorted)
  # 데이터 프레임을 리스트 형태로 변환
  order_list = renamed.to_dict(orient='records')
  return order_list

In [ ]:
# 날짜 계산 함수
def day_diff(day1, day2):
  date_1 = datetime.fromisoformat(day1)
  date_2 = datetime.fromisoformat(day2)

  # 차이 계산
  diff = date_2 - date_1

  # 일 수 차이 출력
  day_diff = diff.days
  return day_diff

#### 코스 추천 알고리즘 함수화

In [ ]:
def time_to_minutes(iso_time: str) -> int:
    # ISO 형식에서 시간 부분 추출
    time_part = iso_time.split('T')[1]  # '23:28:00.000'
    hour, minute = map(int, time_part.split(':')[:2])

    # 시간을 분 단위로 변환
    total_minutes = hour * 60 + minute
    return total_minutes

# 코스 추천 알고리즘 함수화
def Course(df, raw_start_time, raw_end_time, duration): # time은 리스트임
  course_results = []
  check = Course_algs(df)
  for i in range(duration):
    print(f"시작:{raw_start_time[i]}, 끝:{raw_end_time[i]}")
    start_time = time_to_minutes(raw_start_time[i]) # 수정
    end_time = time_to_minutes(raw_end_time[i]) # 수정
    print(f"시작시간{start_time}, 끝시간{end_time}")
    result = check.uniformCostSearch(i, start_time, end_time, 1, 1)
    optimal_tour, start_end_list = result
    if result is None:
      course_results.append({
          f"tour{i}" : {
              "error": "No valid path found."
          }
      })
    else:
      if i == duration - 1: # 마지막 날에는 숙소 방문하지 않으니 제외
        optimal_tour = optimal_tour[:-1]
        start_end_list = start_end_list[:-1]
      course_results.append({
             f"tour{i}" : {
              "Optimal_Tour": optimal_tour,
              "start_end_list": start_end_list
             }
         })
  return course_results

# 분 단위 시간을 "시간:분" 형식으로 반환하는 함수
def convert_minutes_to_time_format(minutes):
    hours = int(minutes // 60)
    remaining_minutes = int(minutes % 60)
    return f"{hours:02}:{remaining_minutes:02}"

# 리스트를 df로 변경하는 함수
def Course_to_dataframe(course_results):
    data = []

    for i, tour in enumerate(course_results):
        tour_name = f"tour{i}"
        tour_info = tour[tour_name]

        if "error" not in tour_info:
            optimal_tour = tour_info["Optimal_Tour"]
            start_end_list = tour_info["start_end_list"]

            for order, (visit_area, (start, end)) in enumerate(zip(optimal_tour, start_end_list)):
                data.append({
                    "VISIT_AREA_NM": visit_area,
                    "I": i,
                    "ORDER": order,
                    "START": convert_minutes_to_time_format(start),
                    "END": convert_minutes_to_time_format(end)
                })
        else:
            # If there is an error, we can skip adding entries or handle it accordingly
            pass

    # Convert the list of dictionaries to a pandas DataFrame
    df_results = pd.DataFrame(data)
    return df_results

def course_result(df, start_time, end_time, duration):
  course_results = Course(df, start_time, end_time, duration)
  course_df = Course_to_dataframe(course_results)
  course_final_df = pd.merge(course_df, df, on='VISIT_AREA_NM', how='left')
  return course_final_df

## 예시

In [ ]:
# # 호출 예시
# values = [1, 2, 7, 2, 6]
# check = recommend(values, places)
# check

In [ ]:
# time_df = df[['VISIT_AREA_NM', 'AVG_TIME_MIN']].drop_duplicates()
# time_df.reset_index(drop = True, inplace = True)

# test_df = pd.merge(check, time_df, on = 'VISIT_AREA_NM', how = 'left')
# test_df

In [ ]:
# test_df.to_csv('Test/test_160_time.csv', index = False)

# 서버

## mission과 motive dictionary

In [ ]:
mis_dict = {'쇼핑':1,
    '테마파크/놀이시설':2,
    '역사 유적지방문':3,
    '시티투어':4,
    '야외스포츠,레포츠':5,
    '지역 문화예술/공연/전시':6,
    '유흥/오락':7,
    '캠핑':8,
    '지역 축제/이벤트 참가':9,
    '온천/스파':10,
    '교육/체험 프로그램 참여':11,
    '드라마 촬영지 방문':12,
    '종교/성지 순례':13,
    'Well-ness여행':21,
    'SNS인생샷 여행':22,
    '호캉스여행':23,
    '신규 여행지 발굴':24,
    '반려동물 동반 여행':25,
    '인플루언서 따라하기':26,
    '친환경 여행':27,
    '등반 여행':28}
motiv_dict = {'일상적인 환경':1,
    '쉴 수 있는 기회':2,
    '여행 동반자와 친밀':3,
    '진정한 자아 찾기':4,
    'SNS 사진 등록':5,
    '운동, 건강증진':6,
    '새로운 경험 추구':7,
    '역사탐방,문화적 경험':8,
    '특별한 목적':9,
    '기타':10}

## App

In [ ]:
# 파일에서 인증 토큰 읽어와 환경 변수 설정
!export NGROK_AUTH_TOKEN=$(cat "Secret/Token/NGROK authtoken.txt")

# ngrok 명령어 실행
!ngrok authtoken $NGROK_AUTH_TOKEN

In [ ]:
app = Flask(__name__)

# /list_recommend 결과를 저장
recommended_df  = pd.DataFrame() # list_recommend
start = ""
end = ""

# 데이터 받고 추천결과 전송
@app.route('/list_recommend', methods=['POST'])
def list_recommend():
  # global 변수 선언
  global recommended_df
  global start, end

  # 초기화
  recommended_df = pd.DataFrame() # 새로운 추천을 필요로 할 때 초기화
  #클라이언트가 서버로 보낸 데이터를 파이썬 딕셔너리로 파싱
  requested_data = request.json
  print("--------------------------------------------------------------------------------------------------------------------------")
  print("[List Recommend Process]")
  print(f"Received Data: {requested_data}", "\n")
  print(f"Dataframe: {recommended_df}", "\n")
  # string을 int로 바꿈
  data_list = [value for value in requested_data.values()]
  data_list[0] = mis_dict[data_list[0]]
  data_list[4] = motiv_dict[data_list[4]]

  # 여행 시작 시간, 종료 시간, 그리고 기간 저장
  start = data_list[5]
  end = data_list[6]
  print("Start: ", start, ", End: ", end)

  ## 모든 요소를 정수형으로 변환
  data_list = list(map(int, data_list[:5]))

  print("Ready ... (0) Complete! \n")

  # 추천결과 저장 및 코스 추천 알고리즘에 입력값 전달
  merge = recommend(data_list, places)

  print("Recommend End ... (9) Complete! \n")

  # 최종 df를 슬라이스하여 카테고리별 상위 5개 리스트 생성
  candidates_df = merge
  candidates_list = candidates_df.to_dict('records')

  #평균 체류시간 데이터 병합
  time_df = df[['VISIT_AREA_NM', 'AVG_TIME_MIN']].drop_duplicates()
  time_df.reset_index(drop = True, inplace = True)
  time_merge = pd.merge(candidates_df, time_df, on = 'VISIT_AREA_NM', how = 'left')

  # 데이터 프레임 글로벌 변수로 저장
  recommended_df = time_merge

  print("Generate Candidate List With Residence Average Time ... (10) Complete! \n")

  # 데이터: (숙소, 식당, 카페, 여행지) -> 전송: (식당, 카페, 여행지, 호텔)
  def create_type_list(start_index, end_index):
        return [
            {
                'name': row['VISIT_AREA_NM'],
                'address': row['ADDR'],
                'satisfaction': row['SCORE'],
                'latitude': row['Y_COORD'],
                'longitude': row['X_COORD']
            }
            for row in candidates_list[start_index:end_index]
        ]

  response_data = {
        '식당': create_type_list(40, 45),
        '카페': create_type_list(80, 85),
        '여행지': create_type_list(120, 125),
        '호텔': create_type_list(0, 5)
    }

  print("Response Data: ", response_data, '\n')
  print("--------------------------------------------------------------------------------------------------------------------------")
  return jsonify(response_data), 200

# 추천 동선 전송
@app.route('/course_recommend', methods=['POST'])
def course_recommend():
  # duration 생성
  duration = day_diff(start[0], end[-1]) + 1
  # 들어온 데이터 확인
  print(f"Dataframe: {recommended_df}", "\n")
  print("Start: ", start, ", End: ", end, ", Duration: ", duration, "\n")

  # 코스 추천 알고리즘으로 결과값 저장
  result_df = course_result(recommended_df, start, end, duration)

  # i 이름 변경
  result = result_df.rename(columns = {'I':'DAY'})

  # ORDER 에 맞춰 list로 변환
  course_result_list = []
  for i in range(duration):
      course_result_list.append({
          i: sort_by_day(i, result)
      })

  # return jsonify(course_result)
  print("Recommend Course: ", course_result_list, "\n")
  print("--------------------------------------------------------------------------------------------------------------------------")
  return jsonify(course_result_list), 200

# ngrok 터널 생성
if __name__ == '__main__':
    # ngrok을 사용하여 서버를 외부에 노출

    public_url = ngrok.connect(5000)
    print('Public URL:', public_url)
    app.run(port=5000)